# Social Post Generator with LangChain
Cairo social-media agency helper: **Prompt Template → structured JSON (Pydantic) → sequential chain (Arabic caption)**.

Free Colab + free Gemini API key (`gemini-3.8-flash`). Before running: add your key in Colab **Secrets** (🔑 icon) with the name `GEMINI_API_KEY` and enable *Notebook access*.

In [1]:
# Cell 1 - install pinned libraries
!pip install -q langchain==1.4.4 langchain-google-genai==4.4.1 pydantic==2.14.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.1/487.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 21.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


## Requirement 1 - Prompt Template & first call

In [2]:
import os
from google import genai
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = userdata.get("GEMINI_API_KEY")   # key comes from Colab Secrets
os.environ["GOOGLE_API_KEY"] = API_KEY

def pick_working_model(preferred="gemini-3.8-flash"):
    """Try the preferred model, then every other available flash model, until one answers."""
    client = genai.Client(api_key=API_KEY)
    names = [m.name.replace("models/", "") for m in client.models.list()
             if "generateContent" in (m.supported_actions or [])]
    skip = ("tts", "image", "audio", "live", "embedding", "robotics", "computer", "learnlm")
    cands = [n for n in names if "flash" in n and not any(s in n for s in skip)]
    cands = sorted(cands, key=lambda n: (n != preferred, "lite" not in n, n))
    for name in cands:
        try:
            ChatGoogleGenerativeAI(model=name).invoke("Say OK")
            print("Using model:", name)
            return name
        except Exception as e:
            print("Skipped", name, "->", str(e)[:70].replace("\n", " "))
    raise RuntimeError("No available model. Create a key from a NEW project in aistudio.google.com.")

MODEL_NAME = pick_working_model()
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0.7)

Skipped gemini-3.8-flash -> Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOU
Skipped gemini-2.5-flash-lite -> Error calling model 'gemini-2.5-flash-lite' (NOT_FOUND): 404 NOT_FOUND
Using model: gemini-3.1-flash-lite


In [3]:
# Cell 3 - ChatPromptTemplate with ONE variable: {description}
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a creative social-media writer based in Cairo, Egypt. You write polished, "
     "friendly posts for local cafes and shops. Rules: title max 60 characters; caption max "
     "280 characters; 3 to 5 hashtags, each starting with #; add a short alt-text that "
     "describes the image for accessibility."),
    ("human",
     "Write an Instagram post draft for this product:\n{description}"),
])

# Cell 4 - first real call: prompt | model, print raw text
raw_chain = prompt | llm

test_description = "new iced karak chai with cardamom, served cold in a tall glass"
raw_response = raw_chain.invoke({"description": test_description})
print(raw_response.content)

[{'type': 'text', 'text': '**Title: Cool Down with Our New Iced Karak! 🧊**\n\nBeat the Cairo heat with a twist! ️ Our signature Karak Chai is now served ice-cold with a fragrant hint of premium cardamom. It’s the perfect refreshing pick-me-up for your afternoon. Come grab a tall glass today! ☕✨\n\n#IcedKarak #CairoEats #KarakChai #CairoCafes\n\n**Alt-text:** A tall glass filled with creamy, light-brown iced karak tea, garnished with a green cardamom pod, sitting on a wooden table.', 'extras': {'signature': 'EnMKcQFpFH0TQ9e80VGWMEDO2Us/HfTaU8FfnDAs1GPqhjO2I2Vk2HOh9k39IeAYOlEjCRH6EzIhZ9ADf+BzKGNT5k0shH3w5LxlMRBtrkCHThbAL5p2oXi+/mxgyPU6WWe996By58KGMvO1+epWlwt/wFQV'}}]


## Requirement 2 - Structured output with Pydantic

In [4]:
# Cell 5 - SocialPost schema
from pydantic import BaseModel, Field, field_validator

class SocialPost(BaseModel):
    title: str = Field(description="Punchy post title, max 60 characters")
    caption: str = Field(description="Engaging caption, max 280 characters")
    hashtags: list[str] = Field(description="3 to 5 hashtags, each starting with #")
    alt_text: str = Field(description="Short accessibility description of the image")

    @field_validator("hashtags")
    @classmethod
    def _fix_hashtags(cls, tags):
        # safety net: make sure every tag starts with '#' and has no spaces
        return [t if t.startswith("#") else "#" + t.replace(" ", "") for t in tags]

In [5]:
# Cell 6 - bind schema to the model and rebuild the chain
structured_llm = llm.with_structured_output(SocialPost)
structured_chain = prompt | structured_llm

descriptions = [
    "new iced karak chai with cardamom, served cold in a tall glass",
    "handmade leather laptop sleeve in three colors, stitched by local artisans in Cairo",
    "weekend special: warm koshari bowl with extra crispy onions and spicy tomato sauce",
]

for d in descriptions:
    post = structured_chain.invoke({"description": d})
    print("INPUT:", d)
    print(post)
    print("Is SocialPost instance?", isinstance(post, SocialPost), "| type:", type(post).__name__)
    print(f"title={len(post.title)} chars, caption={len(post.caption)} chars, hashtags={len(post.hashtags)}")
    print("-" * 60)

INPUT: new iced karak chai with cardamom, served cold in a tall glass
title='Beat the Heat with Iced Karak' caption='Cool down the Cairo way! Our signature Karak Chai is now served over ice with a refreshing hint of aromatic cardamom. The perfect afternoon pick-me-up in a tall, chilled glass. Come grab yours today!' hashtags=['#KarakChai', '#CairoEats', '#IcedTea', '#CairoCafes', '#SummerVibes'] alt_text='A tall glass filled with creamy iced karak chai garnished with green cardamom pods on a wooden table.'
Is SocialPost instance? True | type: SocialPost
title=29 chars, caption=199 chars, hashtags=5
------------------------------------------------------------
INPUT: handmade leather laptop sleeve in three colors, stitched by local artisans in Cairo
title="Crafted in Cairo: Your Laptop's New Best Friend" caption='Elevate your daily carry with our handmade leather laptop sleeves. Skillfully stitched by local artisans right here in Cairo, available in three timeless colors. Durable, stylis

## Requirement 3 - Sequential chain: add Arabic translation

In [6]:
# Cell 7 - translation chain + sequential flow + generate_post()
import time
import pandas as pd
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

translate_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a professional English-to-Arabic translator for social media. Translate the "
     "caption into natural, friendly Arabic suitable for a Cairo audience. Keep emojis. "
     "Return ONLY the Arabic translation, nothing else."),
    ("human", "{caption}"),
])
translate_chain = translate_prompt | llm

def to_text(msg):
    # AIMessage.content can be a str or a list of parts depending on the model/version
    c = msg.content
    if isinstance(c, list):
        c = "".join(p if isinstance(p, str) else p.get("text", "") for p in c)
    return c.strip()

# Step 1: description -> SocialPost   Step 2: post.caption -> Arabic caption
sequential_chain = (
    structured_chain
    | RunnableLambda(lambda post: {"post": post, "caption": post.caption})
    | RunnablePassthrough.assign(arabic=translate_chain)
)

def generate_post(description: str) -> dict:
    out = sequential_chain.invoke({"description": description})
    result = out["post"].model_dump()
    result["arabic_caption"] = to_text(out["arabic"])
    return result

In [7]:
# Cell 8 - run on the same 3 descriptions and save posts.csv
rows = []
for d in descriptions:
    r = generate_post(d)
    rows.append({
        "description": d,
        "title": r["title"],
        "caption": r["caption"],
        "hashtags": " ".join(r["hashtags"]),
        "alt_text": r["alt_text"],
        "arabic_caption": r["arabic_caption"],
    })
    time.sleep(2)  # be gentle with the free-tier rate limit

df = pd.DataFrame(rows, columns=["description","title","caption","hashtags","alt_text","arabic_caption"])
df.to_csv("posts.csv", index=False, encoding="utf-8-sig")  # utf-8-sig so Excel shows Arabic correctly

assert len(df) == 3 and (df["arabic_caption"].str.len() > 0).all()
assert df["hashtags"].apply(lambda s: all(t.startswith("#") for t in s.split())).all()
df

,description,title,caption,hashtags,alt_text,arabic_caption
0,"new iced karak chai with cardamom, served cold...",Cool Down with Our New Iced Karak,Beat the Cairo heat with a twist! 🧊 Our signat...,#KarakChai #CairoEats #IcedTea #CairoFoodies,A tall glass filled with iced karak chai garni...,واجه حر القاهرة بطريقة جديدة! 🧊 شاي الكرك المم...
1,handmade leather laptop sleeve in three colors...,Elevate Your Workday with Cairo Craftsmanship,Carry your laptop in style with our new handma...,#CairoCrafts #HandmadeLeather #SupportLocal #C...,"Three handmade leather laptop sleeves in tan, ...",شيل اللاب توب بتاعك بشياكة مع جرابات الجلد الج...
2,weekend special: warm koshari bowl with extra ...,Weekend Comfort: Our Signature Koshari,Nothing says Cairo weekend like a steaming bow...,#Koshari #CairoEats #WeekendVibes #EgyptianFoo...,A close-up overhead shot of a warm bowl of Kos...,مفيش أحلى من طبق كشري سخن يظبط الويك إند في ال...


In [8]:
# Cell 9 - download the CSV to your computer
from google.colab import files
files.download("posts.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>